# Elevation, in pure Python

`peaknav.terrain` answers *how high is this point* from the same compressed ASTER
dataset the app renders. No renderer, no Java, no display — one dependency (Pillow) and
a network connection the first time each tile is needed.

Tiles are cached after the first download, so the second question about the same area is
answered locally.

In [ ]:
from peaknav.terrain import elevation_at

elevation_at(45.9417, 7.7480)   # the Breithorn, above Zermatt

## Spire summits read low, and it is worth knowing why

ASTER is a stereo DEM: each pixel is about 30 m across, and a sharp rock spire simply
does not fill one. The elevation that comes back is real ground, but for a spire it is
ground *near* the summit rather than the summit itself — so treat these as lower bounds.
Broad summits read true to a few metres.

In [ ]:
peaks = [
    ("Breithorn",    45.9417,  7.7480, 4164),   # broad snow dome
    ("Matterhorn",   45.9763,  7.6586, 4478),   # spire
    ("Mont Blanc",   45.8326,  6.8652, 4808),   # broad
    ("Ben Nevis",    56.7969, -5.0036, 1345),
    ("Mount Rainier", 46.8523, -121.7603, 4392),
]

for name, lat, lon, surveyed in peaks:
    measured = elevation_at(lat, lon)
    print(f"{name:<14} {measured:>5} m   (surveyed {surveyed} m, "
          f"{measured - surveyed:+d})")

## A profile along a line

Sampling between two points is just a loop — the interesting part is that after the
first call the tiles are local, so a hundred samples cost nothing extra.

In [ ]:
def profile(start, end, samples=40):
    """Elevations along a straight line in lat/lon, which is close enough over a few km."""
    (lat1, lon1), (lat2, lon2) = start, end
    for i in range(samples + 1):
        t = i / samples
        lat = lat1 + (lat2 - lat1) * t
        lon = lon1 + (lon2 - lon1) * t
        yield lat, lon, elevation_at(lat, lon)


# Zermatt village up to the Matterhorn.
points = list(profile((46.0207, 7.7491), (45.9763, 7.6586), samples=40))
low = min(p[2] for p in points)
high = max(p[2] for p in points)
for lat, lon, metres in points:
    bar = "#" * int(40 * (metres - low) / max(1, high - low))
    print(f"{metres:>5} m |{bar}")

If matplotlib happens to be installed, the same data plots in one line. It is not a
dependency of `peaknav`, so this cell is allowed to be skipped.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("matplotlib not installed - skipping the plot (pip install matplotlib)")
else:
    plt.figure(figsize=(9, 3))
    plt.plot([p[2] for p in points])
    plt.title("Zermatt to the Matterhorn")
    plt.ylabel("metres")
    plt.xlabel("sample")
    plt.grid(alpha=0.3)
    plt.show()

## Underneath

The dataset is a slippy-map tree of tiles, and the elevation is encoded across two
images. `tile_xy` says which tile covers a point; `decode_elevation` turns a pair of
encoded values back into metres. Both are exported so the encoding is inspectable rather
than magic.

In [ ]:
from peaknav.terrain import DATASET_URL, decode_elevation, tile_xy

print("tile covering the Matterhorn:", tile_xy(45.9763, 7.6586))
print("dataset:", DATASET_URL[:70], "...")

# The PNG sample names a 1024 m band (128 is the one starting at sea level) and the JPEG
# the position inside it, in 4 m steps - flipped in odd-numbered bands.
print("decode_elevation(0, 128)   =", decode_elevation(0, 128), "m   (sea level)")
print("decode_elevation(100, 128) =", decode_elevation(100, 128), "m   (100 steps of 4 m)")
print("decode_elevation(161, 129) =", decode_elevation(161, 129), "m   (band 1, flipped)")

### Ocean, and other places with no tile

Where the dataset has no tile at all — open sea — the answer is 0 rather than an error,
which keeps a sweep over a coastline from having to catch exceptions.

In [ ]:
elevation_at(0.0, -30.0)    # middle of the Atlantic